# Double Inverted Pendulum: PPO (continuous control)

Same MuJoCo double-pendulum swing-up + balance task as the DQN notebook, but the
cart force is now a **continuous** action learned by PPO instead of 7 discretized bins.
Two things were the actual bottleneck for DQN, not the physics: (1) a coarse discrete
action set can't do the fine, small corrections balance needs while still allowing full
force for swing-up, and (2) epsilon-greedy exploration essentially never stumbles into the
long, coordinated energy-pumping sequence swing-up requires. PPO fixes (1) directly with a
continuous Gaussian policy, and helps with (2) only a little (on-policy stochastic actions,
not undirected epsilon-random) -- if swing-up still doesn't emerge, look at reward shaping
or a curriculum before blaming the algorithm again.

MuJoCo stores the second hinge relative to pole 1. The agent observes absolute angles:
`theta1 = q1` and `theta2 = q1 + q2_relative`.


## 1. Prepare Colab
Mount Drive, install headless dependencies, and report the JAX accelerator.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

try:
    from google.colab import drive
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True
    drive.mount("/content/drive")

IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
if IN_COLAB or IN_KAGGLE:
    subprocess.run([
        sys.executable, "-m", "pip", "install", "--upgrade", "-q",
        "mujoco>=3.2", "gymnasium>=1.0", "pandas>=2.0", "matplotlib>=3.8",
        "imageio>=2.34", "imageio-ffmpeg>=0.5", "scipy>=1.11",
    ], check=True)

# Native MuJoCo dynamics are CPU-based; EGL uses the hosted NVIDIA GPU for video rendering.
os.environ.setdefault("MUJOCO_GL", "egl")
print("Runtime:", "Colab" if IN_COLAB else "Kaggle" if IN_KAGGLE else "local")


## 2. Tuning And Persistent Storage
Set `OUTPUT_DIR` to Google Drive. A new run directory is required after changing network dimensions.

`NUM_ENVS` python environments are stepped sequentially each iteration and their transitions batched
into one rollout of length `ROLLOUT_STEPS` per env before every update -- this is what replaces DQN's
replay buffer. Total env-steps per update = `NUM_ENVS * ROLLOUT_STEPS`. This task's swing-up needs a much
larger sample budget than the DQN run got; 2M is a starting point, not a proven-sufficient number -- watch
the dashboard and extend `TOTAL_STEPS` (just re-run the training cell, it resumes from checkpoint) if
`both_upright` hasn't started climbing yet.


In [ ]:
OUTPUT_DIR = Path("/content/drive/MyDrive/ProjectsRuns/TIPy/runs/double/ppo/run-001") if IN_COLAB else Path.cwd() / "double-ppo-run"
CHECKPOINT_DIR, METRICS_PATH = OUTPUT_DIR / "checkpoints", OUTPUT_DIR / "metrics.csv"
DASHBOARD_PATH = OUTPUT_DIR / "dashboard.png"

SEED = 42
TOTAL_STEPS = 2_000_000
MAX_EPISODE_STEPS = 5000
ACTION_LIMIT = 100.0
NUM_ENVS = 16
ROLLOUT_STEPS = 1024
LEARNING_RATE = 3e-4
GAMMA, GAE_LAMBDA = 0.99, 0.95
CLIP_EPS, VALUE_COEF, ENTROPY_COEF = 0.2, 0.5, 0.005
PPO_EPOCHS, NUM_MINIBATCHES = 10, 8
MAX_GRAD_NORM = 0.5
CHECKPOINT_EVERY_UPDATES = 5
KEEP_LAST_CHECKPOINTS = 3
SMOKE_TEST = False
if SMOKE_TEST:
    TOTAL_STEPS, MAX_EPISODE_STEPS = 20_000, 200
    NUM_ENVS, ROLLOUT_STEPS, CHECKPOINT_EVERY_UPDATES = 4, 64, 1

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print("Output:", OUTPUT_DIR)
print("Env-steps per update:", NUM_ENVS * ROLLOUT_STEPS)


## 3. Double-Pendulum MuJoCo Environment

Same model, reward, and observation as the DQN notebook. The only change is the action space:
`Box(-1, 1, shape=(1,))`, linearly scaled to +-`ACTION_LIMIT` newtons, instead of 7 discrete bins.
Rail contact is a true terminal; reaching the time limit is only truncation and still permits
value bootstrapping (handled in the rollout collection below, section 6).


In [ ]:
import gymnasium as gym
from gymnasium import spaces
import mujoco
import numpy as np

MODEL_XML = r"""
<mujoco model="cartpole_double">
  <compiler angle="radian" autolimits="true" inertiafromgeom="false" />
  <option timestep="0.005" gravity="0 0 -9.81" integrator="RK4" />
  <default>
    <geom contype="0" conaffinity="0" />
  </default>
  <visual>
    <headlight diffuse="0.7 0.7 0.7" ambient="0.3 0.3 0.3" specular="0.1 0.1 0.1" />
    <rgba haze="0.15 0.2 0.25 1" />
  </visual>
  <asset>
    <material name="floor_mat" rgba="0.16 0.22 0.28 1" />
    <material name="metal_mat" rgba="0.65 0.68 0.72 1" />
    <material name="cart_mat" rgba="0.82 0.18 0.18 1" />
    <material name="pole1_mat" rgba="0.2 0.75 0.3 1" />
    <material name="pole2_mat" rgba="0.2 0.3 0.75 1" />
    <material name="site_mat" rgba="0.95 0.9 0.15 0.9" />
    <material name="rail_limit_mat" rgba="0.95 0.55 0.05 1" />
  </asset>
  <worldbody>
    <light diffuse="0.7 0.7 0.7" pos="0 0 3.5" dir="0 0 -1" />
    <geom name="floor" type="plane" size="5 5 0.1" material="floor_mat" />
    <body name="frame">
      <geom name="tower_left" type="box" pos="-1.3 0 0.4" size="0.1 0.15 0.8" material="metal_mat" />
      <geom name="tower_right" type="box" pos="1.3 0 0.4" size="0.1 0.15 0.8" material="metal_mat" />
      <geom name="rail" type="box" pos="0 0 0.85" size="2 0.05 0.05" material="metal_mat" />
      <geom name="rail_limit_left" type="box" pos="-2 0 0.95" size="0.025 0.12 0.1" material="rail_limit_mat" />
      <geom name="rail_limit_right" type="box" pos="2 0 0.95" size="0.025 0.12 0.1" material="rail_limit_mat" />
    </body>
    <body name="cart" pos="0 0 1">
      <joint name="cart_slide" type="slide" axis="1 0 0" frictionloss="0" damping="0.1" limited="false" />
      <inertial pos="0 0 0" mass="2" diaginertia="0.0333 0.0333 0.0333" />
      <geom name="cart_geom" type="box" size="0.15 0.08 0.1" mass="2.0" material="cart_mat" />
      <site name="cart_center_site" pos="0 0 0" size="0.02" type="sphere" material="site_mat" />
      <geom name="mount_pin" type="capsule" pos="0 0.095 0" axisangle="1 0 0 1.5708" size="0.015 0.03" material="metal_mat" />
      <body name="pole1" pos="0 0.11 0" quat="6.12323399574e-17 0 -1 0">
        <joint name="pole1_hinge" type="hinge" axis="0 -1 0" frictionloss="0" damping="0.03" ref="3.1415926535897931" limited="false" />
        <inertial pos="0 0 0.15" mass="0.5" diaginertia="0.00376666666667 0.00381666666667 8.33333333333e-05" />
        <site name="pole1_hinge_site" pos="0 0 0" size="0.015" type="sphere" material="site_mat" />
        <geom name="pole1_geom" type="box" pos="0 0 0.15" size="0.02 0.01 0.15" mass="0.5" material="pole1_mat" />
        <site name="pole1_tip_site" pos="0 0 0.3" size="0.015" type="sphere" material="site_mat" />
        <geom name="pole2_mount_pin" type="capsule" pos="0 0.015 0.3" axisangle="1 0 0 1.5708" size="0.012 0.015" material="metal_mat" />
        <body name="pole2" pos="0 0.03 0.3" quat="1 0 -0 0">
          <joint name="pole2_hinge" type="hinge" axis="0 -1 0" frictionloss="0" damping="0.03" ref="0" limited="false" />
          <inertial pos="0 0 0.15" mass="0.5" diaginertia="0.00376666666667 0.00381666666667 8.33333333333e-05" />
          <site name="pole2_hinge_site" pos="0 0 0" size="0.015" type="sphere" material="site_mat" />
          <geom name="pole2_geom" type="box" pos="0 0 0.15" size="0.02 0.01 0.15" mass="0.5" material="pole2_mat" />
          <site name="pole2_tip_site" pos="0 0 0.3" size="0.015" type="sphere" material="site_mat" />
        </body>
      </body>
    </body>
    <camera name="replay" pos="0 6 1.4" fovy="50" xyaxes="-1 0 0 0 -0.15 0.988686" />
  </worldbody>
  <sensor>
    <jointpos name="cart_position" joint="cart_slide" />
    <jointpos name="pole1_relative_angle" joint="pole1_hinge" />
    <jointpos name="pole2_relative_angle" joint="pole2_hinge" />
    <jointvel name="cart_velocity" joint="cart_slide" />
    <jointvel name="pole1_relative_velocity" joint="pole1_hinge" />
    <jointvel name="pole2_relative_velocity" joint="pole2_hinge" />
  </sensor>
  <actuator>
    <motor name="cart_motor" joint="cart_slide" gear="1" ctrlrange="-100 100" forcerange="-100 100" />
  </actuator>
</mujoco>
"""

def wrap_angle(value):
    return float((value + np.pi) % (2 * np.pi) - np.pi)

class DoublePendulumEnv(gym.Env):
    def __init__(self, max_episode_steps=5000, action_limit=100.0):
        super().__init__()
        self.model = mujoco.MjModel.from_xml_string(MODEL_XML)
        self.data = mujoco.MjData(self.model)
        self.max_episode_steps, self.action_limit = max_episode_steps, float(action_limit)
        self.rail_limit, self.dt, self.current_step = 2.0, float(self.model.opt.timestep), 0
        self.action_space = spaces.Box(-1.0, 1.0, shape=(1,), dtype=np.float32)
        self.observation_space = spaces.Box(-1.0, 1.0, shape=(8,), dtype=np.float32)

    def physical_state(self):
        x, relative1, relative2 = map(float, self.data.qpos)
        dx, relative_speed1, relative_speed2 = map(float, self.data.qvel)
        theta1 = wrap_angle(relative1)
        theta2 = wrap_angle(relative1 + relative2)
        dtheta1 = relative_speed1
        dtheta2 = relative_speed1 + relative_speed2
        return x, theta1, theta2, dx, dtheta1, dtheta2

    def observation(self):
        x, theta1, theta2, dx, dtheta1, dtheta2 = self.physical_state()
        return np.array([np.clip(x / self.rail_limit, -1, 1), np.clip(dx / 5, -1, 1),
                         np.cos(theta1), np.sin(theta1), np.cos(theta2), np.sin(theta2),
                         np.clip(dtheta1 / 20, -1, 1), np.clip(dtheta2 / 20, -1, 1)],
                        dtype=np.float32)

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        self.current_step = 0
        mujoco.mj_resetData(self.model, self.data)
        absolute1 = np.pi + self.np_random.uniform(-0.04, 0.04)
        absolute2 = np.pi + self.np_random.uniform(-0.04, 0.04)
        self.data.qpos[:] = [self.np_random.uniform(-0.01, 0.01), absolute1, absolute2 - absolute1]
        absolute_speed1 = self.np_random.uniform(-0.02, 0.02)
        absolute_speed2 = self.np_random.uniform(-0.02, 0.02)
        self.data.qvel[:] = [self.np_random.uniform(-0.01, 0.01), absolute_speed1,
                             absolute_speed2 - absolute_speed1]
        mujoco.mj_forward(self.model, self.data)
        return self.observation(), {}

    def step(self, action):
        self.current_step += 1
        force = float(np.clip(action[0], -1.0, 1.0)) * self.action_limit
        self.data.ctrl[0] = force
        mujoco.mj_step(self.model, self.data)
        x, theta1, theta2, dx, dtheta1, dtheta2 = self.physical_state()
        angle_reward = 0.5 * (np.cos(theta1) + np.cos(theta2))
        reward = (angle_reward - 0.25 * (x / self.rail_limit) ** 2 - 0.01 * dx ** 2
                  - 0.003 * (dtheta1 ** 2 + dtheta2 ** 2)
                  - 0.001 * (force / self.action_limit) ** 2
                  + (3.0 if abs(theta1) < 0.35 and abs(theta2) < 0.35 else 0.0))
        terminated = bool(abs(x) >= self.rail_limit)
        if terminated:
            reward -= 100.0
        truncated = bool(self.current_step >= self.max_episode_steps)
        info = {"x": x, "theta1": theta1, "theta2": theta2, "dx": dx,
                "dtheta1": dtheta1, "dtheta2": dtheta2, "force": force}
        return self.observation(), float(reward), terminated, truncated, info

check_env = DoublePendulumEnv(max_episode_steps=10)
first, _ = check_env.reset(seed=7)
second, _ = check_env.reset(seed=7)
assert first.shape == (8,) and np.allclose(first, second)
assert first[2] < -0.95 and first[4] < -0.95
assert check_env.action_space.shape == (1,)
print("Environment check passed:", first)


## 4. Vectorized Rollout Buffer

`NUM_ENVS` independent Python `DoublePendulumEnv` instances are stepped in a plain loop each
iteration (MuJoCo itself doesn't run on the accelerator, only the network does, so this stays a
host-side loop). This plays the role DQN's replay buffer played: instead of storing a huge pool of
past transitions and sampling from it, PPO only ever trains on the most recent `ROLLOUT_STEPS`
per env, then throws the batch away (on-policy).

**Truncation bootstrap**: when an episode ends only because it hit `MAX_EPISODE_STEPS` (truncated,
not terminated), the true next state still has value -- the agent didn't fail, the clock ran out.
Before that env resets, we add `GAMMA * V(true_next_obs)` onto the stored reward so GAE still
bootstraps correctly across the reset boundary. Rail termination gets no such correction: it's a
real terminal state and its target value is masked to zero, same as in the DQN notebook.


In [ ]:
class VecEnv:
    def __init__(self, num_envs, max_episode_steps, action_limit, base_seed):
        self.envs = [DoublePendulumEnv(max_episode_steps, action_limit) for _ in range(num_envs)]
        self.num_envs = num_envs
        self.episode_returns = np.zeros(num_envs, dtype=np.float32)
        self.episode_lengths = np.zeros(num_envs, dtype=np.int32)
        self.episode_upright = np.zeros(num_envs, dtype=bool)
        self.obs = np.stack([env.reset(seed=base_seed + i)[0] for i, env in enumerate(self.envs)])
        self._reset_counter = base_seed + num_envs

    def step(self, actions, agent, params, key, gamma):
        next_obs = np.empty_like(self.obs)
        rewards = np.empty(self.num_envs, dtype=np.float32)
        terminals = np.empty(self.num_envs, dtype=np.float32)
        finished_episodes = []
        for i, env in enumerate(self.envs):
            obs_i, reward, terminated, truncated, info = env.step(actions[i])
            self.episode_returns[i] += reward
            self.episode_lengths[i] += 1
            self.episode_upright[i] |= abs(info["theta1"]) < 0.35 and abs(info["theta2"]) < 0.35
            if truncated and not terminated:
                _, _, bootstrap_value = agent.act(params, jnp.asarray(obs_i)[None], key)
                reward += gamma * float(bootstrap_value[0])
            rewards[i], terminals[i] = reward, float(terminated)
            if terminated or truncated:
                finished_episodes.append((self.episode_returns[i], self.episode_lengths[i],
                                          self.episode_upright[i]))
                self.episode_returns[i] = self.episode_lengths[i] = 0
                self.episode_upright[i] = False
                self._reset_counter += 1
                obs_i, _ = env.reset(seed=self._reset_counter)
            next_obs[i] = obs_i
        self.obs = next_obs
        return rewards, terminals, finished_episodes

def compute_gae(rewards, values, bootstrap_values, terminals, gamma, lam):
    # `terminals` masks true failure only -- truncation was already folded into `rewards` upstream.
    steps, num_envs = rewards.shape
    advantages = np.zeros_like(rewards)
    running = np.zeros(num_envs, dtype=np.float32)
    for t in reversed(range(steps)):
        mask = 1.0 - terminals[t]
        delta = rewards[t] + gamma * bootstrap_values[t] * mask - values[t]
        running = delta + gamma * lam * mask * running
        advantages[t] = running
    return advantages, advantages + values


## 5. Actor-Critic Network

Diagonal Gaussian policy with a state-independent log-std (standard for continuous-control PPO --
no tanh squashing, actions are clipped to [-1, 1] in `env.step`). Same 256-256 trunk size as the
DQN notebook's Q-network, now feeding both a policy head and a value head.


In [ ]:
import flax.linen as nn

class ActorCritic(nn.Module):
    action_dim: int

    @nn.compact
    def __call__(self, observation):
        trunk = nn.tanh(nn.Dense(256)(observation))
        trunk = nn.tanh(nn.Dense(256)(trunk))
        mean = nn.Dense(self.action_dim, kernel_init=nn.initializers.orthogonal(0.01))(trunk)
        log_std = self.param("log_std", nn.initializers.constant(-0.5), (self.action_dim,))
        value = nn.Dense(1, kernel_init=nn.initializers.orthogonal(1.0))(trunk)
        return mean, log_std, jnp.squeeze(value, -1)

def gaussian_log_prob(mean, log_std, actions):
    std = jnp.exp(log_std)
    return jnp.sum(-0.5 * (((actions - mean) / std) ** 2 + 2 * log_std + jnp.log(2 * jnp.pi)), axis=-1)


## 6. PPO Agent

`act` samples from the Gaussian policy (used during rollout collection; greedy evaluation later
uses the mean directly, no sampling). `update` is the clipped surrogate objective plus a value
loss and an entropy bonus, run for `PPO_EPOCHS` passes over `NUM_MINIBATCHES` shuffled minibatches
of the current rollout -- there is no persistent replay buffer to checkpoint, unlike DQN.


In [ ]:
import functools

class PPOAgent:
    def __init__(self, obs_dim, action_dim, seed):
        self.action_dim, self.rng = int(action_dim), np.random.default_rng(seed)
        self.network = ActorCritic(action_dim)
        self.online = self.network.init(jax.random.PRNGKey(seed), jnp.zeros((1, obs_dim)))
        self.optimizer = optax.chain(optax.clip_by_global_norm(MAX_GRAD_NORM), optax.adam(LEARNING_RATE))
        self.optimizer_state = self.optimizer.init(self.online)
        self.act = jax.jit(self._act)
        self.update_jit = jax.jit(self._update)

    def _act(self, params, obs, key):
        mean, log_std, value = self.network.apply(params, obs)
        std = jnp.exp(log_std)
        action = mean + std * jax.random.normal(key, mean.shape)
        log_prob = gaussian_log_prob(mean, log_std, action)
        return action, log_prob, value

    def act_greedy(self, params, obs):
        mean, _, _ = self.network.apply(params, obs)
        return mean

    def _update(self, params, optimizer_state, batch):
        def loss_function(p):
            mean, log_std, value = self.network.apply(p, batch["obs"])
            log_prob = gaussian_log_prob(mean, log_std, batch["actions"])
            ratio = jnp.exp(log_prob - batch["old_log_probs"])
            advantages = batch["advantages"]
            unclipped = ratio * advantages
            clipped = jnp.clip(ratio, 1 - CLIP_EPS, 1 + CLIP_EPS) * advantages
            policy_loss = -jnp.mean(jnp.minimum(unclipped, clipped))
            value_loss = jnp.mean((value - batch["returns"]) ** 2)
            entropy = jnp.mean(jnp.sum(log_std + 0.5 * jnp.log(2 * jnp.pi * jnp.e), axis=-1))
            total = policy_loss + VALUE_COEF * value_loss - ENTROPY_COEF * entropy
            return total, (policy_loss, value_loss, entropy)
        (loss, aux), grads = jax.value_and_grad(loss_function, has_aux=True)(params)
        updates, optimizer_state = self.optimizer.update(grads, optimizer_state, params)
        return optax.apply_updates(params, updates), optimizer_state, loss, aux

    def state_dict(self):
        return {"online": jax.device_get(self.online),
                "optimizer_state": jax.device_get(self.optimizer_state),
                "rng": self.rng.bit_generator.state}

    def load_state_dict(self, state):
        self.online = jax.tree.map(jnp.asarray, state["online"])
        self.optimizer_state = jax.tree.map(jnp.asarray, state["optimizer_state"])
        self.rng.bit_generator.state = state["rng"]


## 7. Atomic Checkpoints And Metrics
Resume restores the network, optimizer, and random state. Unlike DQN there's no replay buffer to
restore -- each update only ever uses the rollout it just collected.


In [ ]:
import csv
from datetime import datetime, timezone
import os
import pickle
import time

METRIC_FIELDS = ["timestamp", "update", "total_steps", "reward", "policy_loss", "value_loss",
                 "entropy", "episode_length", "steps_per_second", "both_upright"]

def save_checkpoint(agent, update, total_steps):
    payload = {"version": 1, "update": int(update), "total_steps": int(total_steps),
               "agent": agent.state_dict()}
    path = CHECKPOINT_DIR / f"checkpoint_{total_steps:012d}.pkl"
    temporary = path.with_suffix(".tmp")
    with temporary.open("wb") as file:
        pickle.dump(payload, file, pickle.HIGHEST_PROTOCOL)
        file.flush()
        os.fsync(file.fileno())
    temporary.replace(path)
    for old in sorted(CHECKPOINT_DIR.glob("checkpoint_*.pkl"))[:-KEEP_LAST_CHECKPOINTS]:
        old.unlink()
    return path

def restore_checkpoint(agent):
    for path in sorted(CHECKPOINT_DIR.glob("checkpoint_*.pkl"), reverse=True):
        try:
            with path.open("rb") as file:
                payload = pickle.load(file)
            if payload.get("version") != 1:
                raise ValueError("Unsupported checkpoint version")
            agent.load_state_dict(payload["agent"])
            print("Resumed", path.name)
            return int(payload["update"]), int(payload["total_steps"])
        except Exception as error:
            print("Skipped", path.name, error)
    return 0, 0

def append_metrics(row):
    new_file = not METRICS_PATH.exists()
    with METRICS_PATH.open("a", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=METRIC_FIELDS)
        if new_file:
            writer.writeheader()
        writer.writerow(row)
        file.flush()
        os.fsync(file.fileno())


## 8. Train Or Resume
Each update: collect `ROLLOUT_STEPS` steps across `NUM_ENVS` envs, compute GAE, then run
`PPO_EPOCHS` shuffled minibatch passes over that rollout. The first update compiles JAX and will
be slower. Checkpointing occurs every `CHECKPOINT_EVERY_UPDATES` updates.


In [ ]:
vec_env = VecEnv(NUM_ENVS, MAX_EPISODE_STEPS, ACTION_LIMIT, base_seed=SEED)
agent = PPOAgent(8, 1, SEED)
update_index, total_steps = restore_checkpoint(agent)
key = jax.random.PRNGKey(SEED + 1)
session_start, session_start_steps = time.perf_counter(), total_steps
recent_episode_returns, recent_upright = [], []

try:
    while total_steps < TOTAL_STEPS:
        update_index += 1
        obs_buf = np.zeros((ROLLOUT_STEPS, NUM_ENVS, 8), np.float32)
        act_buf = np.zeros((ROLLOUT_STEPS, NUM_ENVS, 1), np.float32)
        logp_buf = np.zeros((ROLLOUT_STEPS, NUM_ENVS), np.float32)
        val_buf = np.zeros((ROLLOUT_STEPS, NUM_ENVS), np.float32)
        rew_buf = np.zeros((ROLLOUT_STEPS, NUM_ENVS), np.float32)
        term_buf = np.zeros((ROLLOUT_STEPS, NUM_ENVS), np.float32)
        episode_lengths_this_update = []

        for t in range(ROLLOUT_STEPS):
            key, subkey = jax.random.split(key)
            actions, log_probs, values = agent.act(agent.online, jnp.asarray(vec_env.obs), subkey)
            actions_np = np.asarray(actions)
            obs_buf[t], act_buf[t] = vec_env.obs, actions_np
            logp_buf[t], val_buf[t] = np.asarray(log_probs), np.asarray(values)
            rewards, terminals, finished = vec_env.step(actions_np, agent, agent.online, subkey, GAMMA)
            rew_buf[t], term_buf[t] = rewards, terminals
            total_steps += NUM_ENVS
            for episode_return, episode_length, upright in finished:
                recent_episode_returns.append(episode_return)
                recent_upright.append(float(upright))
                episode_lengths_this_update.append(episode_length)
            if total_steps >= TOTAL_STEPS:
                break

        _, _, next_values = agent.act(agent.online, jnp.asarray(vec_env.obs), key)
        bootstrap_values = np.concatenate([val_buf[1:], np.asarray(next_values)[None]], axis=0)
        advantages, returns = compute_gae(rew_buf, val_buf, bootstrap_values, term_buf, GAMMA, GAE_LAMBDA)
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

        flat = {"obs": obs_buf.reshape(-1, 8), "actions": act_buf.reshape(-1, 1),
                "old_log_probs": logp_buf.reshape(-1), "advantages": advantages.reshape(-1),
                "returns": returns.reshape(-1)}
        num_samples = flat["obs"].shape[0]
        policy_losses, value_losses, entropies = [], [], []
        for _ in range(PPO_EPOCHS):
            permutation = agent.rng.permutation(num_samples)
            for minibatch_indices in np.array_split(permutation, NUM_MINIBATCHES):
                batch = {k: jnp.asarray(v[minibatch_indices]) for k, v in flat.items()}
                agent.online, agent.optimizer_state, loss, aux = agent.update_jit(
                    agent.online, agent.optimizer_state, batch)
                policy_losses.append(float(aux[0]))
                value_losses.append(float(aux[1]))
                entropies.append(float(aux[2]))

        elapsed = max(time.perf_counter() - session_start, 1e-9)
        append_metrics({
            "timestamp": datetime.now(timezone.utc).isoformat(), "update": update_index,
            "total_steps": total_steps,
            "reward": np.mean(recent_episode_returns[-50:]) if recent_episode_returns else np.nan,
            "policy_loss": np.mean(policy_losses), "value_loss": np.mean(value_losses),
            "entropy": np.mean(entropies),
            "episode_length": np.mean(episode_lengths_this_update) if episode_lengths_this_update else np.nan,
            "steps_per_second": (total_steps - session_start_steps) / elapsed,
            "both_upright": np.mean(recent_upright[-50:]) if recent_upright else 0.0,
        })
        if update_index % CHECKPOINT_EVERY_UPDATES == 0:
            print("Saved", save_checkpoint(agent, update_index, total_steps))
        print(update_index, total_steps,
              round(np.mean(recent_episode_returns[-50:]) if recent_episode_returns else float("nan"), 1),
              "entropy", round(np.mean(entropies), 3))
finally:
    save_checkpoint(agent, update_index, total_steps)


## 9. Static Training Dashboard
`both_upright` here is a rolling fraction over the last 50 finished episodes, not a single episode's
flag as in the DQN notebook -- with many envs finishing episodes at different times per update, a
rolling rate reads more cleanly than a per-update boolean.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

metrics = pd.read_csv(METRICS_PATH).drop_duplicates("update", keep="last").sort_values("update")
window = min(20, len(metrics))
x = metrics["update"]  # NOTE: `metrics.update` shadows DataFrame.update(); always index with ["update"]
figure, axes = plt.subplots(2, 3, figsize=(16, 9), constrained_layout=True)
figure.suptitle("TIPy Double Pendulum PPO", fontsize=16, fontweight="bold")
axes[0, 0].plot(x, metrics.reward, alpha=0.3)
axes[0, 0].plot(x, metrics.reward.rolling(window, min_periods=1).mean())
axes[0, 0].set_title("Reward (rolling mean of recent episodes)")
axes[0, 1].plot(x, metrics.policy_loss, label="policy")
axes[0, 1].plot(x, metrics.value_loss, label="value")
axes[0, 1].legend(); axes[0, 1].set_title("PPO losses")
axes[0, 2].plot(x, metrics.entropy); axes[0, 2].set_title("Policy entropy")
axes[1, 0].plot(x, metrics.episode_length); axes[1, 0].set_title("Episode length")
axes[1, 1].plot(x, metrics.steps_per_second); axes[1, 1].set_title("Steps/second")
axes[1, 2].plot(x, metrics.both_upright); axes[1, 2].set_title("Both-upright rate")
for axis in axes.flat:
    axis.set_xlabel("Update")
    axis.grid(alpha=0.25)
figure.savefig(DASHBOARD_PATH, dpi=160)
plt.show()
print("Saved", DASHBOARD_PATH)


## 10. Greedy Evaluation
Evaluation uses the policy mean directly (no sampling) and performs no learning.


In [ ]:
evaluation_rewards, captures = [], 0
eval_env = DoublePendulumEnv(MAX_EPISODE_STEPS, ACTION_LIMIT)
for evaluation_episode in range(10):
    observation, _ = eval_env.reset(seed=40_000 + evaluation_episode)
    total_reward, captured = 0.0, False
    while True:
        action = np.asarray(agent.act_greedy(agent.online, jnp.asarray(observation)[None]))[0]
        observation, reward, terminated, truncated, info = eval_env.step(action)
        total_reward += reward
        captured |= abs(info["theta1"]) < 0.35 and abs(info["theta2"]) < 0.35
        if terminated or truncated:
            break
    evaluation_rewards.append(total_reward)
    captures += int(captured)
print("Greedy reward mean/std:", np.mean(evaluation_rewards), np.std(evaluation_rewards))
print("Both-pole capture rate:", captures / len(evaluation_rewards))


## Replay The Trained Run

Colab cannot reliably open MuJoCo's interactive desktop viewer. This block runs a deterministic
evaluation, streams rendered frames directly into an MP4, saves it in the run directory, and
displays it inline. It replays the current trained policy; it is not an exact recording of a
stochastic training episode.


In [ ]:
import imageio.v2 as imageio
from IPython.display import Video, display

REPLAY_SEED, REPLAY_SECONDS, REPLAY_FPS = 50_004, 10.0, 50
REPLAY_PATH = OUTPUT_DIR / "replay.mp4"
replay_env = DoublePendulumEnv(int(REPLAY_SECONDS / 0.005), ACTION_LIMIT)
observation, _ = replay_env.reset(seed=REPLAY_SEED)
renderer = mujoco.Renderer(replay_env.model, height=480, width=640)
writer = imageio.get_writer(REPLAY_PATH, fps=REPLAY_FPS, codec="libx264", quality=8)
frame_stride = max(1, round(1 / (replay_env.dt * REPLAY_FPS)))
try:
    for step in range(replay_env.max_episode_steps):
        action = np.asarray(agent.act_greedy(agent.online, jnp.asarray(observation)[None]))[0]
        observation, _, terminated, truncated, _ = replay_env.step(action)
        if step % frame_stride == 0:
            renderer.update_scene(replay_env.data, camera="replay")
            writer.append_data(renderer.render())
        if terminated or truncated:
            break
finally:
    writer.close()
    renderer.close()
print("Saved replay:", REPLAY_PATH)
display(Video(str(REPLAY_PATH), embed=True))
